<a href="https://colab.research.google.com/github/jleonoras/ollama-remote-gpu/blob/main/Remote_Ollama_GPU_Backend_for_VS_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Remote Ollama GPU Backend for VS Code**

# Step #1. Initialize Environment

In [ ]:
#@markdown <h3>Initialize Environment { display-mode: "form" }

import os, subprocess, sys

# CHECKING RUNTIME TYPE
def check_runtime():
    try:
        # Check if NVIDIA driver is accessible
        subprocess.check_output('nvidia-smi')
        print("✅ T4 GPU DETECTED. Proceeding with optimized setup...")
        return True
    except:
        print("\n" + "!"*40)
        print("❌ GPU NOT DETECTED!")
        print("Please go to: Runtime -> Change runtime type -> Select 'T4 GPU' -> Save.")
        print("Then run this cell again.")
        print("!"*40 + "\n")
        return False

if check_runtime():
    print("📦 Installing dependencies quietly...")

    # Clean install sequence - Hidden outputs for a clean dashboard
    !sudo apt-get update -y &> /dev/null
    !sudo apt-get install -y zstd pciutils &> /dev/null

    # Install Ollama (Showing logs as it's quick and confirms GPU detection)
    print("📥 Installing/Updating Ollama...")
    !curl -fsSL https://ollama.com/install.sh | sh

    # Python API dependencies
    print("🐍 Installing Python libraries...")
    !pip install pyngrok ollama -q

    print("\n" + "="*40)
    print("✅ SYSTEM READY!")
    print("Run the next cell to start the API Tunnel.")
    print("="*40)

# Step #2. Start Ollama API Tunnel

In [ ]:
#@markdown <h3>Start Ollama API Tunnel { display-mode: "form" }

#@markdown > Get your token here: [https://dashboard.ngrok.com/get-started/your-authtoken](https://dashboard.ngrok.com/get-started/your-authtoken)

NGROK_TOKEN = "" #@param {type:"string"}
SELECTED_MODEL = "qwen3-coder-next" #@param ["qwen2.5-coder:7b", "deepseek-coder:6.7b", "llama3.1:8b", "codellama:7b", "mistral"] {allow-input: true}
#@markdown > *Tip: You can also manually type other model names from [ollama.com/library](https://ollama.com/library)*

import os, subprocess, time, requests
from pyngrok import ngrok
from IPython.display import HTML, display, clear_output

# 1. CLEANUP & ENV SETUP
!pkill ollama
try: ngrok.kill()
except: pass

os.environ.update({
    'LD_LIBRARY_PATH': '/usr/lib64-nvidia',
    'OLLAMA_HOST': '0.0.0.0:11434',
    'OLLAMA_ORIGINS': '*',
    'NVIDIA_VISIBLE_DEVICES': 'all'
})

# 2. START OLLAMA
print("🚀 Initializing Ollama Service...")
subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, env=os.environ)

# 3. HEALTH CHECK
for i in range(15):
    try:
        if requests.get("http://localhost:11434/").status_code == 200: break
    except: time.sleep(1)

# 4. SMART CHECKLIST LOGIC
def get_local_models():
    try:
        output = subprocess.check_output(["ollama", "list"]).decode('utf-8')
        return output
    except: return ""

print(f"🔍 Checking storage for {SELECTED_MODEL}...")
local_inventory = get_local_models()

if SELECTED_MODEL in local_inventory:
    status_msg = f"🟢 <b>{SELECTED_MODEL}</b> is ready in storage."
else:
    print(f"📥 {SELECTED_MODEL} not found locally. Pulling from library...")
    subprocess.run(["ollama", "pull", SELECTED_MODEL])
    status_msg = f"✅ <b>{SELECTED_MODEL}</b> has been downloaded."

# 5. FINAL UI REFRESH (CLEAN & INLINE)
try:
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(11434, proto="http").public_url

    clear_output(wait=True)
    ui_html = f"""
    <div style="font-family: 'Segoe UI', sans-serif; font-size: 13.5px; color: #8b949e; margin-top: 15px;">
        <div style="margin-bottom: 10px;">{status_msg}</div>
        <div style="font-family: monospace; font-size: 14px; display: flex; align-items: center; gap: 10px;">
            <b style="color: #58a6ff;">Public URL:</b>
            <a href="{public_url}" target="_blank" style="color: #aff5b4; text-decoration: underline;">{public_url}</a>
            <button onclick="copyToClipboard()" id="copy-btn"
                    style="background: #238636; color: white; border: none; border-radius: 4px; padding: 2px 12px; cursor: pointer; font-size: 11px;">
                Copy Link
            </button>
        </div>
        <p style="margin-top: 8px; font-size: 12px; color: #58a6ff;">Paste this link into VS Code (Continue/Cline settings)</p>
    </div>
    <script>
    function copyToClipboard() {{
        const el = document.createElement('textarea'); el.value = "{public_url}";
        document.body.appendChild(el); el.select(); document.execCommand('copy'); document.body.removeChild(el);
        const btn = document.getElementById("copy-btn"); btn.innerText = "Copied!";
        setTimeout(() => {{ btn.innerText = "Copy Link"; }}, 2000);
    }}
    </script>
    """
    display(HTML(ui_html))
except Exception as e:
    print(f"❌ Ngrok Error: {e}")

# Step #3. Model Management (Check & Delete)

In [ ]:
#@title 🗑️ Model Management { display-mode: "form" }
#@markdown Run this cell to see installed models and delete unwanted ones to save space.

MODEL_TO_DELETE = "" #@param {type:"string"}
CONFIRM_DELETE = False #@param {type:"boolean"}

import subprocess

def list_models():
    print("📋 CURRENT INSTALLED MODELS:")
    print("-" * 50)
    !ollama list
    print("-" * 50)

if MODEL_TO_DELETE and CONFIRM_DELETE:
    print(f"🔥 Deleting model: {MODEL_TO_DELETE}...")
    !ollama rm {MODEL_TO_DELETE}
    print("✅ Done!")
    list_models()
elif MODEL_TO_DELETE and not CONFIRM_DELETE:
    print("⚠️ Please check 'CONFIRM_DELETE' to proceed with removal.")
    list_models()
else:
    list_models()